# Predicting Next-Iteration Eqsat Memory

Compare models that predict the next iteration's memory from the current
iteration's egraph state, rule applications, timings, and live heap.

`generate.py` records per-iteration measurements for terms in
`data/seed_terms/*/terms.json`.

The evaluation reports absolute next memory and memory growth. Cross-validation
is grouped by seed term.

In [ ]:
import altair as alt
import polars as pl

import iteration_data as D
import memory_model as M
import memory_plots as MP
import plots as P

alt.theme.register("analysis", enable=True)(lambda: P.THEME)

## Load the iteration traces

Load the latest seed folder into one row per iteration. Rewrite counts are
stored in `rule_<name>` columns.

In [ ]:
SEED_DIR = D.resolve_seed_dir()

iterations = D.load_iterations(SEED_DIR)
iterations.select(
    "term_size", "iter_index", "egraph_nodes", "egraph_classes", "allocated", "n_rebuilds"
).head()

## Build the supervised frame

Each row pairs iteration `i` with iteration `i+1` memory. The following rows
are excluded:

- Run ends without a successor.
- Transitions into stop iterations, which execute no rewrites and are not
  comparable with an ordinary next iteration.

Derived features use only current and previous iterations.

In [ ]:
transitions = D.build_transitions(iterations)

SCALARS, RULES = D.feature_columns(transitions)
FEATURES = SCALARS + RULES
print(f"{len(SCALARS)} scalar features, {len(RULES)} rewrite-rule features")

transitions.select(
    "term_size", "iter_index", "egraph_nodes", "allocated", "next_allocated", "y_log_growth"
).head()

## Target distribution

Distribution of the log memory growth ratio.

In [ ]:
MP.growth_histogram(transitions)

## Cross-validated model comparison

Three predictors are evaluated with 5-fold `GroupKFold` by seed term:

- **naive (carry forward):** current memory
- **ridge:** log-scaled features
- **gradient boosting:** raw features

`median error ×` is the exponentiated median absolute log error.

In [ ]:
metrics, predictions = M.evaluate(transitions, FEATURES, RULES)
metrics

In [ ]:
MP.metric_bars(metrics, metric="R2")

In [ ]:
MP.predicted_vs_actual(predictions, "log memory growth ratio")

In [ ]:
MP.residual_distribution(predictions, "log memory growth ratio")

Residuals by egraph size.

In [ ]:
MP.residual_vs_size(predictions, "log memory growth ratio")

## Permutation importance

The boosted model is fitted on four group folds and evaluated on the fifth.

In [ ]:
importance = M.importances(transitions, FEATURES, RULES, target="y_log_growth")
MP.importance_bars(importance)

## History window

`build_transitions(..., window=n)` adds the previous `n - 1` scalar values and
the log deltas between consecutive lags.

Lags are computed before filtering. Missing warm-up lags use the first
iteration in the run. Rule counts include only the current iteration.

In [ ]:
sweep = M.window_sweep(iterations, windows=(1, 2, 3, 4, 6, 8))
sweep.filter(pl.col("target") == "log memory growth ratio").select(
    "window", "model", "R2", "MAE (log)", "median error x", "n_features"
).sort("model", "window")

In [ ]:
MP.window_sweep_chart(sweep, metric="R2")

## Extrapolating to larger terms

Train below a split chosen from the observed sizes and test on the largest
quarter of size levels.

In [ ]:
term_sizes = sorted(transitions["term_size"].unique().to_list())
SPLIT_SIZE = term_sizes[-max(1, len(term_sizes) // 4)]
print(f"Training below size {SPLIT_SIZE}; testing at and above it")
M.size_extrapolation(transitions, FEATURES, RULES, split_size=SPLIT_SIZE)

## Scope

- Prediction horizon: one iteration
- Data source: one seed folder and rewrite language
- Stop iterations: excluded as targets